# Notebook 08: Production Assembly

**MoEForecaster**: a Mixture-of-Experts (MoE) model for hourly electricity demand forecasting in Colombia's National Interconnected System (SIN).

---

## Objective

Assemble the final MoE system for production:

1. Retrain every expert on all available data (2020 and 2021 excluded), using the model family selected in Notebook 05 and the hyperparameters found in Notebook 07, subject to a complexity constraint based on cluster size.
2. Retrain the day-type and week-type classifiers selected in Notebook 04 on all available data; they drive the gating at prediction time.
3. Wrap experts, classifiers and gating parameters in the `MoEForecaster` class (defined in `src/moe_forecaster/forecaster.py`).
4. Run an in-sample sanity check over the last week of data and a recursive demonstration forecast for January 2026.
5. Serialize the complete system (`moe_forecaster.pkl`) together with its components, configuration files and a model card, and verify that the serialized model can be reloaded and used.

## Inputs

| File | Produced by | Content |
|---|---|---|
| `data/processed/features.parquet` | Notebook 02 | Hourly feature matrix and target `demand` |
| `data/clustering/cluster_assignments.parquet` | Notebook 03 | Hourly cluster assignments (`cluster_day`, `cluster_week`, `expert_id`) |
| `results/05_expert_selection_best.csv` | Notebook 05 | Selected model family per `expert_id` |
| `models/hyperparameters/best_hyperparameters.json` | Notebook 07 | Best hyperparameters per expert |
| `models/hyperparameters/best_gating_params.json` | Notebook 07 | Gating threshold and temperature |
| `models/classifiers/best_classifier_day.pkl`, `models/classifiers/best_classifier_week.pkl` | Notebook 04 | Selected day-type and week-type classifiers |

If the HPO results, the gating parameters or the classifiers are not found, the notebook falls back to the default values defined in the code (see Sections 2.5 and 3.3).

## Outputs

All production artifacts are written to `models/production/`:

| File | Content |
|---|---|
| `moe_forecaster.pkl` | Complete serialized `MoEForecaster` object (used by Notebook 09) |
| `experts/expert_<expert_id>.pkl` | One fitted model per expert |
| `classifier_day.pkl`, `classifier_week.pkl` | Retrained classifiers with their label encoders and feature lists |
| `gating_params.json` | Gating threshold, temperature and minimum soft-gating probability |
| `normalization_params.json` | Reference statistics of the demand per expert (mean, standard deviation, 5th and 95th percentiles, sample size) |
| `feature_config.json` | Feature configuration (feature lists, maximum lag, horizon) |
| `model_card.json` | Model card (architecture, data, features, constraints, references, artifact list) |

Figures written to `results/`: `08_fig_sanity_test.png` and `08_fig_model_architecture.png`.

## Place in the pipeline

01 data acquisition → 02 preprocessing and feature engineering → 03 clustering → 04 cluster analysis and classifiers → 05 expert selection → 06 gating design → 07 hyperparameter optimization → **08 production assembly** → 09 benchmark.

Notebook 09 loads `models/production/moe_forecaster.pkl` to compare the MoE against the reference models.

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

## 1. Configuration and imports

Random seed, number of parallel jobs, model version, imports, plotting style and the production folders `models/production/` and `models/production/experts/`.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
RANDOM_SEED = 42
N_JOBS      = -1

MODEL_VERSION = "1.0.0"   # change with every major retraining

import os, json, pickle, warnings
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

import holidays

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.fontsize': 10, 'axes.grid': True, 'grid.alpha': 0.3,
})

# Production folders (created by the environment setup cell)
PROD_DIR = Path(MODELS_DIR) / 'production'
EXPERTS_DIR = PROD_DIR / 'experts'

print(f'Production folder: {PROD_DIR}')

---
## 2. Load the complete data set and the artifacts

### 2.1 Feature matrix

The hourly feature matrix of Notebook 02 is loaded and sorted chronologically. If it is missing, the notebook stops: no synthetic fallback data are used.

In [ ]:
# ============================================================
# Load the complete data set
# ============================================================
def load_artifact(paths, description='artifact'):
    """Return the first existing path among the candidate paths, or None."""
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f'  ✓ {description} found at: {p}')
            return p
    print(f'  ⚠️  {description} not found')
    return None

# ---- Features ----
print('Loading data...')
features_path = load_artifact(
    [f'{DATA_DIR}/processed/features.parquet'],
    'features.parquet'
)
if features_path:
    df = pd.read_parquet(features_path)
    if not isinstance(df.index, pd.DatetimeIndex):
        for col in ['timestamp','datetime','date']:
            if col in df.columns:
                df = df.set_index(col); break
    df = df.sort_index()
else:
    # No synthetic fallback data: if an artifact from a previous notebook is
    # missing, that notebook must be run first.
    raise FileNotFoundError(f"{DATA_DIR}/processed/features.parquet not found. Run Notebook 02 first (make pipeline).")

### 2.2 Exclusion of 2020 and 2021

The years 2020 and 2021 are removed, as in Notebooks 04 to 07 and 09, so that the production experts are trained on the same period used for model selection and HPO. This period coincides with the COVID-19 pandemic.

In [ ]:
df = df[~df.index.year.isin([2020, 2021])]

In [ ]:
df.head()

### 2.3 Cluster assignments

The hourly cluster assignments of Notebook 03 are indexed by their own timestamp and joined with the feature matrix (inner join). The columns `cluster_day` and `cluster_week` are renamed to `type_day` and `type_week` (day type and week type); `expert_id` identifies the (day type, week type) combination.

In [ ]:
# ---- Clusters ----
cluster_path = load_artifact(
    [f'{DATA_DIR}/clustering/cluster_assignments.parquet',
     f'{DATA_DIR}/processed/cluster_assignments.parquet'],
    'cluster_assignments.parquet'
)
if cluster_path:
    clusters = pd.read_parquet(cluster_path)
    if not isinstance(clusters.index, pd.DatetimeIndex):
        for col in ['timestamp','datetime','date']:
            if col in clusters.columns:
                clusters = clusters.set_index(col); break

clusters = clusters.rename(columns={'cluster_day': 'type_day', 'cluster_week': 'type_week'})

# Each assignment carries its own timestamp (datetime column, see Notebook 03). Rebuilding it
# from the row position shifted it by 48 h, because the clustering drops the first incomplete ISO week.
clusters['timestamp'] = clusters.index
clusters = clusters.reset_index(drop=True)

print(f"\nCluster table shape: {clusters.shape}")
print(f"Cluster table columns: {list(clusters.columns)}")

# Make sure the timestamp is a datetime
clusters['timestamp'] = pd.to_datetime(clusters['timestamp'])

clusters.index = clusters['timestamp']
clusters.drop(columns=['timestamp'], inplace=True)

In [ ]:
clusters

In [ ]:
# Inner join: keep only the hours that have both features and a cluster assignment
df = df.join(clusters[["expert_id", "type_day", "type_week"]], how="inner")

### 2.4 Feature set

The same feature set as in Notebooks 05 to 07 is used: calendar and holiday features plus the autoregressive demand lags (`demand_lag_*`). The target is `demand`.

In [ ]:
# Feature set: calendar/holiday features + demand lags
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(df.columns)

# Lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

In [ ]:
TARGET = 'demand'
EXCLUDE = [TARGET, 'type_day', 'type_week', 'expert_id']
# FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]
FEATURE_COLS = calendar_lags

print(f'\nComplete data set: {df.shape}')
print(f'Range: {df.index[0].date()} → {df.index[-1].date()}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Unique experts: {df["expert_id"].nunique()}')

expert_ids = sorted(df['expert_id'].dropna().unique())
print(f'\nExperts: {expert_ids}')

In [ ]:
df.head()

### 2.5 HPO results, expert selection and gating parameters

- The best hyperparameters per expert (Notebook 07) are read from `models/hyperparameters/best_hyperparameters.json`. That file is keyed by the position of the expert in the list of `expert_id`s sorted lexicographically, and is therefore accessed below with `expert_ids.index(eid)`.
- The model family selected per `expert_id` (Notebook 05) is read from `results/05_expert_selection_best.csv`.
- The gating threshold and temperature (Notebook 07) are read from `models/hyperparameters/best_gating_params.json`. If this file is missing, the defaults `threshold = 0.75` and `tau = 1.0` are used.

In [ ]:
# ============================================================
# Load the best HPO parameters (Notebook 07)
# ============================================================

best_params_global = {}
hp_path = load_artifact(
    [f'{MODELS_DIR}/hyperparameters/best_hyperparameters.json'],
    'best_hyperparameters.json'
)

if hp_path:
    with open(hp_path) as f:
        best_params_global = json.load(f)
    print(f'HPO parameters loaded: {len(best_params_global)} experts')
else:
    print('  No HPO parameters: the default values of the model factory will be used')

# Load the model selection (Notebook 05)
selection_path = load_artifact(
    [f'{RESULTS_DIR}/05_expert_selection_best.csv'],
    'expert_selection.csv'
)
expert_selection = {}
if selection_path:
    df_sel = pd.read_csv(selection_path)
    if 'expert_id' in df_sel.columns and 'model' in df_sel.columns:
        expert_selection = dict(zip(df_sel['expert_id'], df_sel['model']))
    print(f'Selection loaded: {len(expert_selection)} experts with an assigned model')

# Load the gating parameters (Notebook 07)
gating_path = load_artifact(
    [f'{MODELS_DIR}/hyperparameters/best_gating_params.json'],
    'gating_params.json'
)
if gating_path:
    with open(gating_path) as f:
        gating_params = json.load(f)
else:
    gating_params = {'threshold': 0.75, 'tau': 1.0}
    print(f'  Using defaults: {gating_params}')

THRESHOLD = gating_params.get('threshold', 0.75)
TAU       = gating_params.get('tau', 1.0)
print(f'\nGating: threshold={THRESHOLD}, tau={TAU}')

---
## 3. Retraining on the full data set

### 3.1 Model factory: model choice per expert

The model of each expert is chosen by the function `get_model_for_expert` according to the number of training samples $n$ of its cluster. Small clusters are restricted to strongly regularized linear models to limit variance; larger clusters use the family selected in Notebook 05:

| Cluster size | Model | Rationale |
|---|---|---|
| $n < 50$ | Standardization + Ridge (default $\alpha = 100$) | High bias, minimal variance |
| $50 \le n < 200$ | Standardization + ElasticNet (default $\alpha = 10$, `l1_ratio` = 0.5) | Sufficient regularization |
| $n \ge 200$ | Family selected in Notebook 05: BayesianRidge, Lasso, Ridge, XGBoost or Random Forest; any other family name (or a missing selection) defaults to LightGBM | Enough data for more complex models |

The HPO parameters of Notebook 07 are looked up by their unprefixed names (e.g. `alpha`, `n_estimators`, `learning_rate`); any parameter that is not found falls back to the default value written in the code. `BayesianRidge` is always built with its default parameters.

In [ ]:
# ============================================================
# Model factory
# ============================================================

def get_model_for_expert(expert_id, n_samples, best_params=None, selected_model=None):
    """
    Return the (unfitted) model for an expert, based on:
    1. The model family selected in Notebook 05 (if available)
    2. The best HPO parameters from Notebook 07 (if available)
    3. A complexity constraint based on the cluster size
    """
    params = best_params or {}

    # Size-based constraint
    if n_samples < 50:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', Ridge(alpha=params.get('alpha', 100.0), random_state=RANDOM_SEED))
        ])
    elif n_samples < 200:
        alpha = params.get('alpha', 10.0)
        l1_ratio = params.get('l1_ratio', 0.5)
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=2000))
        ])
        return lgb.LGBMRegressor(**lgbm_params)
    else:
        # Large cluster: model family selected in Notebook 05
        model_name = selected_model or 'lgbm'

        if 'bayesian_ridge' in model_name.lower():
            return Pipeline([
                ('scaler', StandardScaler()),
                ('model', BayesianRidge())
            ])
        elif 'lasso' in model_name.lower():
            return Pipeline([
                ('scaler', StandardScaler()),
                ('model', Lasso(alpha=params.get('alpha', 1.0), random_state=RANDOM_SEED))
            ])
        elif 'ridge' in model_name.lower():
            return Pipeline([
                ('scaler', StandardScaler()),
                ('model', Ridge(alpha=params.get('alpha', 1.0), random_state=RANDOM_SEED))
            ])
        elif 'xgb' in model_name.lower():
            xgb_params = {
                'n_estimators': params.get('n_estimators', 500),
                'learning_rate': params.get('learning_rate', 0.05),
                'max_depth': params.get('max_depth', 6),
                'subsample': params.get('subsample', 0.8),
                'colsample_bytree': params.get('colsample_bytree', 0.8),
                'reg_alpha': params.get('reg_alpha', 0.1),
                'reg_lambda': params.get('reg_lambda', 1.0),
                'random_state': RANDOM_SEED, 'verbosity': 0, 'n_jobs': N_JOBS,
            }
            return xgb.XGBRegressor(**xgb_params)
        elif 'random_forest' in model_name.lower() or 'rf' in model_name.lower():
            return RandomForestRegressor(
                n_estimators=params.get('n_estimators', 300),
                max_depth=params.get('max_depth', None),
                min_samples_leaf=params.get('min_samples_leaf', 5),
                random_state=RANDOM_SEED, n_jobs=N_JOBS,
            )
        else:
            # Default: LightGBM
            lgbm_params = {
                'n_estimators': params.get('n_estimators', 500),
                'learning_rate': params.get('learning_rate', 0.05),
                'num_leaves': params.get('num_leaves', 63),
                'min_child_samples': params.get('min_child_samples', 10),
                'subsample': params.get('subsample', 0.8),
                'colsample_bytree': params.get('colsample_bytree', 0.8),
                'reg_alpha': params.get('reg_alpha', 0.01),
                'reg_lambda': params.get('reg_lambda', 0.1),
                'random_state': RANDOM_SEED, 'verbose': -1, 'n_jobs': N_JOBS,
            }
            return lgb.LGBMRegressor(**lgbm_params)

print('Model factory ready.')
print('Complexity constraints:')
print('  n <  50  → Ridge')
print('  n < 200  → ElasticNet')
print('  n ≥ 200  → model selected in Notebook 05 (with Notebook 07 HPO parameters)')

### 3.2 Training the experts

Rows with missing values in the features or the target are dropped (`df_clean`). Each expert is then fitted on all hourly samples of its cluster; experts with fewer than 10 samples are skipped. The training log records the sample size, the resulting model class and whether HPO parameters were available.

In [ ]:
# ============================================================
# Train the K experts on all data
# ============================================================

print('=== TRAINING THE K EXPERTS ON ALL DATA ===')
print(f'Complete data set: {len(df):,} observations\n')

production_experts = {}
expert_training_log = []

df_clean = df.dropna(subset=FEATURE_COLS + [TARGET])

for eid in expert_ids:
    mask = df_clean['expert_id'] == eid
    n_samples = mask.sum()

    if n_samples < 10:
        print(f'  ⚠️  Expert {eid}: only {n_samples} obs, skipped')
        continue

    X_k = df_clean.loc[mask, FEATURE_COLS].values
    y_k = df_clean.loc[mask, TARGET].values

    # HPO parameters (keyed by the position of the expert in the sorted expert_ids list)
    params_k = best_params_global.get(f'{expert_ids.index(eid)}', {})
    model_name_k = expert_selection.get(eid, None)

    print(model_name_k)  # selected family (Notebook 05)
    print(params_k)      # HPO parameters (Notebook 07)

    model_k = get_model_for_expert(eid, n_samples, params_k, model_name_k)
    model_k.fit(X_k, y_k)

    production_experts[eid] = model_k

    # Model family (class of the final estimator)
    if hasattr(model_k, 'steps'):
        family = type(model_k.steps[-1][1]).__name__
    else:
        family = type(model_k).__name__

    expert_training_log.append({
        'expert_id': eid, 'n_samples': n_samples,
        'model_family': family,
        'has_hpo_params': len(params_k) > 0,
    })

    print(f'  ✓ {eid:20s}: n={n_samples:5,}, model={family}')

print(f'\nExperts trained: {len(production_experts)}/{len(expert_ids)}')

df_log = pd.DataFrame(expert_training_log)
display(df_log)

In [ ]:
production_experts

### 3.3 Retraining the day-type and week-type classifiers

At prediction time the cluster of a new hour is unknown, so the gating relies on two classifiers that predict the day type and the week type from the features. The classifiers selected in Notebook 04 (`models/classifiers/best_classifier_{day,week}.pkl`) are loaded and refitted on all available data, using the same feature set as the experts and labels encoded with a `LabelEncoder`. If a classifier is not available, a standardized logistic regression is trained instead.

In [ ]:
# ============================================================
# Retrain the classifiers on all data
# ============================================================

print('=== TRAINING THE CLUSTER CLASSIFIERS ===')

# Classifier features: the same feature set as the experts (calendar/holiday features + demand lags)
CLF_FEATURES = FEATURE_COLS
print(f'Classifier features: {len(CLF_FEATURES)}')

df_clf = df.dropna(subset=CLF_FEATURES + ['type_day', 'type_week'])
X_clf = df_clf[CLF_FEATURES].values

# Day-type classifier (type_day)
from sklearn.preprocessing import LabelEncoder
le_day = LabelEncoder()
y_day = le_day.fit_transform(df_clf['type_day'].astype(str))

# Load the classifier selected in Notebook 04, if it exists
clf_day_path = load_artifact(
    [f'{MODELS_DIR}/classifiers/best_classifier_day.pkl',
     f'{MODELS_DIR}/production/classifier_day.pkl'],
    'best_classifier_day'
)

if clf_day_path:
    with open(clf_day_path, 'rb') as f:
        clf_day_loaded = pickle.load(f)
    # A dict with metadata (Notebook 04 format)
    if isinstance(clf_day_loaded, dict):
        clf_day_obj = clf_day_loaded.get('model', clf_day_loaded.get('pipeline'))
    else:
        clf_day_obj = clf_day_loaded
    if clf_day_obj is not None:
        # Refit on all data
        clf_day_obj.fit(X_clf, y_day)
        clf_day_prod = clf_day_obj
        print('  ✓ classifier_day retrained (Notebook 04 model)')
    else:
        clf_day_prod = None
else:
    clf_day_prod = None

if clf_day_prod is None:
    # Fallback: logistic regression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    clf_day_prod = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, multi_class='multinomial',
                                    C=1.0, random_state=RANDOM_SEED, n_jobs=N_JOBS))
    ])
    clf_day_prod.fit(X_clf, y_day)
    print('  ✓ classifier_day trained (LogisticRegression fallback)')

# Week-type classifier (type_week)
le_week = LabelEncoder()
y_week = le_week.fit_transform(df_clf['type_week'].astype(str))

clf_week_path = load_artifact(
    [f'{MODELS_DIR}/classifiers/best_classifier_week.pkl',
     f'{MODELS_DIR}/production/classifier_week.pkl'],
    'best_classifier_week'
)

if clf_week_path:
    with open(clf_week_path, 'rb') as f:
        clf_week_loaded = pickle.load(f)
    if isinstance(clf_week_loaded, dict):
        clf_week_obj = clf_week_loaded.get('model', clf_week_loaded.get('pipeline'))
    else:
        clf_week_obj = clf_week_loaded
    if clf_week_obj is not None:
        clf_week_obj.fit(X_clf, y_week)
        clf_week_prod = clf_week_obj
        print('  ✓ classifier_week retrained (Notebook 04 model)')
    else:
        clf_week_prod = None
else:
    clf_week_prod = None

if clf_week_prod is None:
    clf_week_prod = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=1.0,
                                    random_state=RANDOM_SEED, n_jobs=N_JOBS))
    ])
    clf_week_prod.fit(X_clf, y_week)
    print('  ✓ classifier_week trained (LogisticRegression fallback)')

print('\nClassifiers ready.')

---
## 4. The `MoEForecaster` class (production wrapper)

`MoEForecaster` encapsulates the complete MoE system behind a scikit-learn-style `predict` method. The class is defined in `src/moe_forecaster/forecaster.py` (not in this notebook), so that the serialized object can be loaded from any script or notebook.

For each hour, the day-type and week-type classifiers output class probabilities, and the confidence is $c = \min(\max_d p_{\text{day}}(d),\, \max_w p_{\text{week}}(w))$:

- **Hard gating** ($c \ge$ `gating_threshold`): the prediction of the expert of the most probable (day type, week type) combination is used.
- **Soft gating** ($c <$ `gating_threshold`): the experts are combined with weights $p_{\text{day}}(d)\,p_{\text{week}}(w)$, keeping only combinations with weight above `min_prob_soft` (0.05), and normalized by the sum of the weights.
- **Fallback**: if the selected expert does not exist, or no expert passes the soft-gating filter, the mean prediction of all experts is used.

`predict_with_metadata` additionally returns, for each hour, the selected expert, the classifier confidences and the gating mode (`hard`, `hard_fallback` or `soft`). The temperature `tau` is stored in the object and in the model card, but the prediction methods of the class do not use it.

The second cell creates the production instance and prints its model card.

In [ ]:
# ============================================================
# MoEForecaster class
# ============================================================
# The class lives in src/moe_forecaster/forecaster.py so that moe_forecaster.pkl
# can be loaded outside this notebook: if it were defined here, pickle would store it
# as __main__.MoEForecaster, which only exists inside this Jupyter session.
# (src/ was added to sys.path by the environment setup cell.)
from moe_forecaster import MoEForecaster
print('MoEForecaster class imported from src/moe_forecaster.')

In [ ]:
# ============================================================
# Create the production instance
# ============================================================

moe_forecaster = MoEForecaster(
    experts=production_experts,
    classifier_day=clf_day_prod,
    classifier_week=clf_week_prod,
    label_encoder_day=le_day,
    label_encoder_week=le_week,
    feature_cols=FEATURE_COLS,
    clf_feature_cols=CLF_FEATURES,
    gating_threshold=THRESHOLD,
    tau=TAU,
    min_prob_soft=0.05,
    version=MODEL_VERSION,
)

card = moe_forecaster.get_model_card()
print('=== MODEL CARD ===')
for k, v in card.items():
    if k != 'expert_ids':
        print(f'  {k}: {v}')
print(f'  expert_ids: {card["expert_ids"]}')

---
## 5. In-sample sanity check (last 168 h)

The production model is applied to the last 500 hours of the training data, using the observed lagged demand as features (one-step-ahead predictions, no recursion). Because these hours were used for training, the resulting MAPE is an **in-sample** value: it only checks that the system is assembled correctly, and is not an estimate of out-of-sample accuracy (see Notebook 09). The checks are:

- predictions within a physically plausible range (3 to 15 million kWh per hour);
- distribution of the gating modes;
- in-sample MAPE below 10%.

The figure shows the last 168 hours (one week): predicted vs. actual demand with a ±3% band, and the percentage error.

In [ ]:
# ============================================================
# In-sample sanity check
# ============================================================

print('=== IN-SAMPLE SANITY CHECK (last 168 h) ===')

# Use the last (up to) 500 observations of the cleaned data set
n_seed = min(500, len(df_clean))
df_seed = df_clean.iloc[-n_seed:]

# One-step-ahead predictions with the observed lags (no recursion in the sanity check)
X_seed = df_seed[FEATURE_COLS].values
y_seed = df_seed[TARGET].values

preds_sanity, metadata_sanity = moe_forecaster.predict_with_metadata(df_seed)

# Range checks
preds_valid = preds_sanity
n_out_of_range = np.sum((preds_valid < 3e6) | (preds_valid > 15e6))
preds_min = preds_valid.min()
preds_max = preds_valid.max()
preds_mean = preds_valid.mean()

print(f'\nPhysical range checks:')
print(f'  Min prediction:   {preds_min:,.0f} kWh  (expected > 3,000,000 kWh)  {"✓" if preds_min > 3e6 else "✗"}')
print(f'  Max prediction:   {preds_max:,.0f} kWh  (expected < 15,000,000 kWh) {"✓" if preds_max < 15e6 else "✗"}')
print(f'  Mean prediction:  {preds_mean:,.0f} kWh  (expected ~7-11 million) {"✓" if 5e6 < preds_mean < 12e6 else "?"}')
print(f'  Out of range:     {n_out_of_range} obs {"✓" if n_out_of_range == 0 else "⚠️"}')

# Gating statistics
mode_counts = metadata_sanity['gating_mode'].value_counts()
print(f'\nGating mode distribution:')
for mode, cnt in mode_counts.items():
    print(f'  {mode}: {cnt} ({100*cnt/len(metadata_sanity):.1f}%)')

# Metrics on the seed window (in-sample: sanity check only)
mape_sanity = 100 * np.mean(np.abs((y_seed - preds_sanity) / (y_seed + 1e-9)))
print(f'\nIn-sample MAPE (seed window): {mape_sanity:.2f}%')
if mape_sanity < 10:
    print('  ✓ The model is learning (in-sample MAPE < 10%)')
else:
    print('  ⚠️  High in-sample MAPE: check data/models')

# Sanity-check figure
n_plot = min(168, len(df_seed))  # 1 week
idx_plot = df_seed.index[-n_plot:]

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Panel 1: prediction vs. actual
ax1 = axes[0]
ax1.plot(idx_plot, y_seed[-n_plot:], 'k-', lw=1.5, label='Actual (XM)', alpha=0.9)
ax1.plot(idx_plot, preds_sanity[-n_plot:], 'b-', lw=1.2, label='MoE prediction', alpha=0.85)
ax1.fill_between(idx_plot,
                  preds_sanity[-n_plot:]*0.97,
                  preds_sanity[-n_plot:]*1.03,
                  alpha=0.2, color='blue', label='±3% band')
ax1.set_ylabel('Demand (kWh)')
ax1.set_title('In-sample Prediction vs. Actual (last week)')
ax1.legend(loc='upper right')

# Panel 2: percentage error
ax2 = axes[1]
error_pct = 100 * (preds_sanity[-n_plot:] - y_seed[-n_plot:]) / (y_seed[-n_plot:] + 1e-9)
ax2.plot(idx_plot, error_pct, color='tomato', lw=0.8, alpha=0.8)
ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.axhline(3, color='orange', lw=0.8, ls=':')
ax2.axhline(-3, color='orange', lw=0.8, ls=':')
ax2.set_ylabel('Error (%)')
ax2.set_xlabel('Date')
ax2.set_title('Percentage error (orange lines = ±3%)')

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR) / '08_fig_sanity_test.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n✓ Sanity check completed.')

---
## 6. Demonstration: recursive forecast for January 2026

This cell produces an hourly forecast for 1–31 January 2026 (744 hours) by **recursive** prediction:

- the calendar and holiday features of each future hour are computed from the timestamp and the Colombian holiday calendar (`holidays.CO`);
- each lag feature `demand_lag_<L>` takes the observed demand at $t - L$ if available, otherwise the forecast already produced for $t - L$; lags that fall in the gap between the end of the data and the forecast window are imputed with the last observed demand;
- each forecast is appended to the series and becomes available as a lag for the following hours.

The forecast is plotted and summarized (minimum, maximum and mean). No observed values are available for this period, so no error metric is computed.

In [ ]:
print("Generating forecasts for January 2026...")

# 1. Forecast window (January 2026)
start_date_pred = pd.Timestamp('2026-01-01 00:00:00')
end_date_pred = pd.Timestamp('2026-01-31 23:00:00')
future_index = pd.date_range(start=start_date_pred, end=end_date_pred, freq='H')

# DataFrame that stores the future features and the forecasts
df_pred_features = pd.DataFrame(index=future_index, columns=FEATURE_COLS + [TARGET])

# Colombian holidays over a relevant range of years
# (earlier years for 'days_from_prev_holiday', later years for 'days_to_next_holiday')
co_holidays = holidays.CO(years=range(2020, 2027))

def generate_features_for_timestamp(ts, historical_df, future_df, feature_cols, target_col, co_holidays):
    """Build the feature vector of a given timestamp.
    historical_df: DataFrame with the observed historical data (up to the end of training)
    future_df: DataFrame with the forecasts already produced for the future (used for lags)
    """
    features = {}

    # 1. Cyclical time features
    features['hour_sin'] = np.sin(2 * np.pi * ts.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * ts.hour / 24)
    features['dayofweek_sin'] = np.sin(2 * np.pi * ts.dayofweek / 7)
    features['dayofweek_cos'] = np.cos(2 * np.pi * ts.dayofweek / 7)
    features['month_sin'] = np.sin(2 * np.pi * ts.month / 12)
    features['month_cos'] = np.cos(2 * np.pi * ts.month / 12)

    # 2. Holiday features
    features['is_holiday'] = 1 if ts.date() in co_holidays else 0

    features['is_bridge'] = 0
    if ts.weekday() == 0 and (ts.date() - timedelta(days=3)) in co_holidays:
        features['is_bridge'] = 1
    if ts.weekday() == 4 and (ts.date() + timedelta(days=3)) in co_holidays:
        features['is_bridge'] = 1
    if (ts.date() - timedelta(days=1)) in co_holidays and (ts.date() + timedelta(days=1)) in co_holidays:
        features['is_bridge'] = 1

    features['days_to_next_holiday'] = 0
    features['days_from_prev_holiday'] = 0

    if ts.date() not in co_holidays:
        next_holiday_date = None
        for day_offset in range(1, 366):
            future_date = ts.date() + timedelta(days=day_offset)
            if future_date in co_holidays:
                next_holiday_date = future_date
                break
        if next_holiday_date:
            features['days_to_next_holiday'] = (next_holiday_date - ts.date()).days
        else:
            features['days_to_next_holiday'] = 365

        prev_holiday_date = None
        for day_offset in range(1, 366):
            past_date = ts.date() - timedelta(days=day_offset)
            if past_date in co_holidays:
                prev_holiday_date = past_date
                break
        if prev_holiday_date:
            features['days_from_prev_holiday'] = (ts.date() - prev_holiday_date).days
        else:
            features['days_from_prev_holiday'] = 365

    # 3. Lag features
    # Make sure the indexes are datetimes for concatenation and lookup
    if not isinstance(historical_df.index, pd.DatetimeIndex):
        historical_df.index = pd.to_datetime(historical_df.index)
    if not isinstance(future_df.index, pd.DatetimeIndex):
        future_df.index = pd.to_datetime(future_df.index)

    # Get the last known real demand for imputation in the initial gap
    last_known_demand = historical_df[target_col].iloc[-1]

    # Combine historical and future data for the lag lookup.
    # historical_df[target_col] contributes only observed demand values;
    # future_df[target_col].dropna() contributes only the forecasts already produced.
    combined_data = pd.concat([historical_df[target_col], future_df[target_col].dropna()]).sort_index()

    lag_cols_nums = sorted([int(c.split('_')[-1]) for c in feature_cols if c.startswith('demand_lag_')])
    for lag in lag_cols_nums:
        lag_ts = ts - timedelta(hours=lag)
        if lag_ts in combined_data.index:
            features[f'demand_lag_{lag}'] = combined_data.loc[lag_ts]
        else:
            # If the lag is not available (e.g., falls in the gap between historical_df and current ts
            # or before df_clean.index.min()), impute with the last known historical demand.
            features[f'demand_lag_{lag}'] = last_known_demand

    # Keep only the features listed in FEATURE_COLS
    return {k: v for k, v in features.items() if k in feature_cols}

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

for i, ts in enumerate(future_index):
    # Build the features of the current timestamp
    current_features_dict = generate_features_for_timestamp(ts, df_clean, df_pred_features, FEATURE_COLS, TARGET, co_holidays)

    # One-row DataFrame for the prediction
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=FEATURE_COLS)

    # Predict
    pred = moe_forecaster.predict(X_current)[0]

    # Store the forecast (it becomes available as a lag for later hours)
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Forecast {i + 1}/{len(future_index)} hours of January 2026. Last forecast: {ts} -> {pred:,.0f} kWh")

print("\nJanuary 2026 forecast completed.")

# Plot the forecast
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Demand forecast (kWh)', color='blue')
plt.title('Electricity Demand Forecast for January 2026')
plt.xlabel('Date')
plt.ylabel('Demand (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Summary statistics of the forecast
preds_jan_2026 = df_pred_features[TARGET].dropna().values
preds_min = preds_jan_2026.min()
preds_max = preds_jan_2026.max()
preds_mean = preds_jan_2026.mean()

print(f'\nForecast statistics for January 2026:')
print(f'  Min forecast:  {preds_min:,.0f} kWh')
print(f'  Max forecast:  {preds_max:,.0f} kWh')
print(f'  Mean forecast: {preds_mean:,.0f} kWh')
print(f'  Valid physical range (3,000-15,000 kWh): {"✓" if 3000 < preds_min and preds_max < 15000 else "✗"}')

---
## 7. Serialization of the complete system

All production artifacts are written to `models/production/`:

1. one pickle file per expert (`experts/expert_<expert_id>.pkl`);
2. the two classifiers, each packaged with its label encoder, feature list and classes;
3. the gating parameters (`gating_params.json`);
4. reference statistics of the demand per expert (`normalization_params.json`);
5. the feature configuration (`feature_config.json`);
6. the model card (`model_card.json`), which includes the test-set metrics if `results/08_evaluation_summary.json` exists;
7. the complete `MoEForecaster` object (`moe_forecaster.pkl`).

In [ ]:
# ============================================================
# Save all production artifacts
# ============================================================

print('=== SERIALIZING THE COMPLETE MoE SYSTEM ===')

# 1. The K individual experts
print('\n[1/6] Individual experts...')
for eid, model in production_experts.items():
    path = EXPERTS_DIR / f'expert_{eid}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'  ✓ {len(production_experts)} experts saved to {EXPERTS_DIR}')

# 2. Classifiers
print('\n[2/6] Classifiers...')
clf_pkg_day = {
    'model': clf_day_prod,
    'label_encoder': le_day,
    'feature_cols': CLF_FEATURES,
    'classes': le_day.classes_.tolist(),
    'n_classes': len(le_day.classes_),
}
with open(PROD_DIR / 'classifier_day.pkl', 'wb') as f:
    pickle.dump(clf_pkg_day, f, protocol=pickle.HIGHEST_PROTOCOL)

clf_pkg_week = {
    'model': clf_week_prod,
    'label_encoder': le_week,
    'feature_cols': CLF_FEATURES,
    'classes': le_week.classes_.tolist(),
    'n_classes': len(le_week.classes_),
}
with open(PROD_DIR / 'classifier_week.pkl', 'wb') as f:
    pickle.dump(clf_pkg_week, f, protocol=pickle.HIGHEST_PROTOCOL)
print('  ✓ classifier_day.pkl and classifier_week.pkl')

# 3. Gating parameters
print('\n[3/6] Gating parameters...')
gating_config = {
    'threshold': THRESHOLD,
    'tau': TAU,
    'min_prob_soft': 0.05,
    'description': 'threshold: hard/soft switching. tau: temperature of the distance softmax.',
}
with open(PROD_DIR / 'gating_params.json', 'w') as f:
    json.dump(gating_config, f, indent=2)
print(f'  ✓ gating_params.json: {gating_config}')

# 4. Normalization parameters (reference demand statistics per cluster)
print('\n[4/6] Normalization params...')
norm_params = {}
for eid in expert_ids:
    mask = df_clean['expert_id'] == eid
    if mask.sum() > 0:
        y_k = df_clean.loc[mask, TARGET]
        norm_params[eid] = {
            'mean': float(y_k.mean()),
            'std': float(y_k.std()),
            'p05': float(y_k.quantile(0.05)),
            'p95': float(y_k.quantile(0.95)),
            'n_samples': int(mask.sum()),
        }
with open(PROD_DIR / 'normalization_params.json', 'w') as f:
    json.dump(norm_params, f, indent=2)
print(f'  ✓ normalization_params.json: {len(norm_params)} clusters')

# 5. Feature config
print('\n[5/6] Feature config...')
lag_cols = [c for c in FEATURE_COLS if c.startswith('demand_lag_')]
rolling_cols = [c for c in FEATURE_COLS if c.startswith('demand_rolling_')]
calendar_cols = [c for c in FEATURE_COLS if any(c.startswith(p) for p in
                ['hour_','dow_','month_','is_holiday','is_bridge','days_to','days_from'])]
fourier_cols = [c for c in FEATURE_COLS if c.startswith('fourier_')]

feature_config = {
    'target': TARGET,
    'all_features': FEATURE_COLS,
    'clf_features': CLF_FEATURES,
    'lag_features': lag_cols,
    'rolling_features': rolling_cols,
    'calendar_features': calendar_cols,
    'fourier_features': fourier_cols,
    'n_features': len(FEATURE_COLS),
    'max_lag': max([int(c.split('_')[-1]) for c in lag_cols]) if lag_cols else 0,
    'forecast_horizon_h': 336,
    'leakage_free': True,
    'rolling_center': False,
}
with open(PROD_DIR / 'feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)
print(f'  ✓ feature_config.json')

# 6. Complete model card
print('\n[6/6] Model card...')

# Load test-set evaluation metrics, if available (no notebook of the pipeline currently writes this file)
eval_summary = {}
eval_path = Path(RESULTS_DIR) / '08_evaluation_summary.json'
if eval_path.exists():
    with open(eval_path) as f:
        eval_summary = json.load(f)

model_card = {
    'model_name': 'MoEForecaster-SIN-Colombia',
    'version': MODEL_VERSION,
    'created_at': datetime.now().isoformat(),
    'description': 'Decoupled Mixture-of-Experts for hourly electricity demand forecasting of the Colombian SIN. Horizon: 336 h (14 days).',
    'architecture': {
        'type': 'Decoupled MoE (cluster-then-predict)',
        'n_experts': len(production_experts),
        'clustering': 'Dual: type_day (24-D daily profiles) × type_week (168-D weekly profiles)',
        'gating': f'Adaptive: hard if min_conf >= {THRESHOLD}, soft if confidence is lower',
        'expert_families': list(set(
            type(m.steps[-1][1]).__name__ if hasattr(m, 'steps') else type(m).__name__
            for m in production_experts.values()
        )),
    },
    'data': {
        'source': 'XM S.A. SINERGOX API (POST https://servapibi.xm.com.co/hourly)',
        'target': 'Actual demand, XM variable DemaReal (kWh), Colombian SIN',
        'training_start': str(df_clean.index[0].date()),
        'training_end': str(df_clean.index[-1].date()),
        'n_obs_total': len(df_clean),
        'resolution': 'hourly',
        'horizon_h': 336,
    },
    'features': feature_config,
    'performance': eval_summary if eval_summary else {
        'note': 'Test-set metrics not available (results/08_evaluation_summary.json not found)'
    },
    'constraints': {
        'no_climate_variables': True,
        'no_future_data': True,
        'leakage_free': True,
        'rolling_center': False,
    },
    'references': [
        'Diebold & Mariano (1995). Comparing Predictive Accuracy. JBES.',
        'Hyndman & Koehler (2006). MASE. International Journal of Forecasting.',
        'Bergstra & Bengio (2012). Random Search for Hyper-Parameter Optimization. JMLR.',
        'Hong & Shahidehpour (2015). Short-term electric load forecasting. Wiley.',
    ],
    'artifacts': {
        'experts': f'models/production/experts/expert_*.pkl ({len(production_experts)} files)',
        'classifier_day': 'models/production/classifier_day.pkl',
        'classifier_week': 'models/production/classifier_week.pkl',
        'gating_params': 'models/production/gating_params.json',
        'normalization_params': 'models/production/normalization_params.json',
        'feature_config': 'models/production/feature_config.json',
    },
}
with open(PROD_DIR / 'model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False)
print('  ✓ model_card.json')

# 7. Complete serialized MoEForecaster
with open(PROD_DIR / 'moe_forecaster.pkl', 'wb') as f:
    pickle.dump(moe_forecaster, f, protocol=pickle.HIGHEST_PROTOCOL)
print('  ✓ moe_forecaster.pkl (complete object)')

---
## 8. Final model overview

The figure shows the distribution of model families across the experts and the number of training samples per expert, with reference lines at n = 50 and n = 200 (the size limits of the model factory) and at n = 1000. A text summary of the production system follows.

In [ ]:
# ============================================================
# Architecture overview and summary
# ============================================================

# ---- Figure: model families and cluster sizes ----
family_counts = {}
for eid, model in production_experts.items():
    if hasattr(model, 'steps'):
        fam = type(model.steps[-1][1]).__name__
    else:
        fam = type(model).__name__
    family_counts[fam] = family_counts.get(fam, 0) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: pie chart of model families
ax1 = axes[0]
colors = plt.cm.Set2(np.linspace(0, 1, len(family_counts)))
wedges, texts, autotexts = ax1.pie(
    family_counts.values(),
    labels=family_counts.keys(),
    autopct='%1.0f%%',
    colors=colors, startangle=90,
)
ax1.set_title(f'Model family distribution\n({len(production_experts)} experts)')

# Panel 2: bar chart of training samples per expert
ax2 = axes[1]
if expert_training_log:
    df_log_plot = df_log.sort_values('n_samples', ascending=False)
    colors2 = ['#2196F3' if n >= 1000 else '#FF9800' if n >= 200 else '#4CAF50' if n >= 50 else '#F44336'
               for n in df_log_plot['n_samples']]
    ax2.barh(df_log_plot['expert_id'], df_log_plot['n_samples'], color=colors2, alpha=0.85)
    ax2.axvline(x=50, color='red', ls='--', lw=0.8, label='n = 50 (Ridge / ElasticNet limit)')
    ax2.axvline(x=200, color='orange', ls='--', lw=0.8, label='n = 200 (ElasticNet / selected family limit)')
    ax2.axvline(x=1000, color='blue', ls='--', lw=0.8, label='n = 1000')
    ax2.set_xlabel('Number of training samples')
    ax2.set_title('Cluster size per expert')
    ax2.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR) / '08_fig_model_architecture.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- Final summary ----
print('\n' + '='*60)
print('MoE SYSTEM: PRODUCTION SUMMARY')
print('='*60)
print(f'Version:            {MODEL_VERSION}')
print(f'Created:            {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Number of experts:  {len(production_experts)}')
print(f'Gating threshold:   {THRESHOLD}')
print(f'Tau:                {TAU}')
print(f'Number of features: {len(FEATURE_COLS)}')
print(f'Horizon:            336 h (14 days)')
print(f'Data source:        XM S.A. SINERGOX API')
print()
print('Model families:')
for fam, cnt in sorted(family_counts.items(), key=lambda x: -x[1]):
    print(f'  {fam:30s}: {cnt} experts')
print()
print('Generated artifacts:')
for name, path in model_card['artifacts'].items():
    print(f'  ✓ {path}')

if eval_summary:
    print(f'\nTest-set performance:')
    print(f'  Global MAPE:           {eval_summary.get("moe_mape_global", "N/A")}%')
    print(f'  Global MASE:           {eval_summary.get("moe_mase_global", "N/A")}')
    print(f'  Improvement vs global: {eval_summary.get("improvement_vs_global_pct", "N/A")}%')
    target_ok = eval_summary.get('moe_mape_global', 99) < 3.0
    print(f'  Target MAPE < 3%:      {"✅ ACHIEVED" if target_ok else "⚠️  PENDING"}')
else:
    print('\n⚠️  No test-set metrics found (results/08_evaluation_summary.json); see Notebook 09 for the benchmark')

print('='*60)

---
## 9. Final check: reload and use the serialized model

`moe_forecaster.pkl` is reloaded from disk and used to predict the last 24 hours of the data set, to verify that the serialized object is self-contained and functional.

In [ ]:
# ============================================================
# FINAL CHECK: load and use the serialized model
# ============================================================

print('=== CHECK: loading moe_forecaster.pkl ===')

with open(PROD_DIR / 'moe_forecaster.pkl', 'rb') as f:
    moe_loaded = pickle.load(f)

# Prediction test
n_test_rows = min(24, len(df_clean))
X_verify = df_clean.iloc[-n_test_rows:]
preds_verify = moe_loaded.predict(X_verify)

print(f'\nVerification predictions (last {n_test_rows} h):')
print(f'  Mean: {preds_verify.mean():,.0f} kWh')
print(f'  Min:  {preds_verify.min():,.0f} kWh')
print(f'  Max:  {preds_verify.max():,.0f} kWh')
print(f'  Valid physical range: {"✓" if 3000 < preds_verify.min() and preds_verify.max() < 15000 else "✗"}')
print(f'\n✓ The serialized model works correctly.')
print('\nMoE system ready for production deployment.')